# Домашна работа по МУ 2025/26

Предвидување дали индустриски клиенти на компанија за одржување ќе ја напуштат компанијата и ќе ги откажат услугите.

Потребно е да се направи модел за предвидување на тоа кои клиенти (индустриски компании) на една компанија за одржување на индустриска опрема би ја напуштиле компанијата и би го раскинале договорот. Ова се прави со цел компанијата да има предвид кои клиенти би требало да се труди да ги задржи пред да се откажат од услугите. На располагање имате податоци за клиенти на компанијата во форма на 18 различни карактеристики.

Вашиот модел треба да ја најде корелацијата помеѓу овие карактеристики за секој клиент и тоа дали клиентот го раскинал договорот или не, и според тоа да е способен да предвиди дали било кој друг клиент даден во соодветно множество за тестирање би ја напуштил компанијата. Можете да користите било кој метод за класификација (без разлика дали е учен на предавања или не), и да користите било кои од дадените карактеристики.

Успешноста на вашите алгоритми ќе биде евалуирана според F1-score, што значи дека се стремите да имате што е можно повисок резултат, при што 1 значи идеално предвидување. Метриката со која ќе бидат распределени поените за домашната ќе биде следната:

_Points_ = _U_ × _[ B + 25 F1 + 5 × (N + 1- P)/N + E]_

каде што: U∈{0,1} е бинарен индикатор дали сте прикачиле решение или не, B∈{0,5} е бинарен индикатор дали решението е подобро од Benchmark-от или не, F1 е точноста на вашето решение според метриката F1, N е вкупниот број на учесници кои прикачиле барем по едно решение, P е вашиот пласман на конечната табела, E∈[0,10] се поените добиени за квалитетот на вашиот елаборат.





In [41]:
import pandas as pd
import numpy as np
import seaborn as sns

import tensorflow as tf
import keras

from keras.models import Sequential, load_model
from keras.layers import Dense, Dropout
from keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from random import seed, randint
import cv2 # conda install -c anaconda opencv

seed(0)

## Вчитување на податоците


In [23]:
data_train = pd.read_csv('IndustrialMaintenanceClientRetention_Training.csv')
data_train.head()

,ID,contract_terminated,distance_to_service_hub,downtime_tolerance,emergency_service_usage,equipment_complexity_level,equipment_health_score,facility_role_type,maintenance_engagement_level,monthly_service_fee,months_on_current_service_tier,months_with_provider,num_prior_providers,onsite_service_satisfaction,overall_contract_satisfaction,preventive_training_sessions_last_year,service_package_tier,staff_skill_level,total_equipment_age,years_operating_facility
0,1,0,3,3,No,1,3,PrecisionManufacturing,2,2177,0,1.0,3.0,3,4.0,6,Silver,3.0,18,45
1,2,0,3,3,No,2,4,PharmaceuticalProduction,4,6842,4,5.0,6.0,4,1.0,3,Silver,2.0,24,22
2,3,0,8,3,No,2,3,HeavyIndustrialProcessing,3,5121,0,0.0,3.0,2,1.0,3,Silver,5.0,18,20
3,4,0,8,3,No,3,3,AdvancedAutomationCenter,2,13757,8,9.0,2.0,4,2.0,5,Silver,1.0,27,26
4,5,1,9,3,No,1,3,PackagingAndFinishing,2,4400,2,3.0,3.0,1,1.0,2,Bronze,2.0,17,11


# Чистење и оптимизирање на податоците 

In [24]:
data_train.dropna(inplace=True)

In [25]:
medians = data_train['num_prior_providers'].median()
medians

np.float64(2.0)

In [26]:
data_train['num_prior_providers'] = data_train['num_prior_providers'].replace(9999999, medians)

In [27]:
data_train['ID'] = data_train['ID'].astype(np.uint16)
data_train['contract_terminated'] = data_train['contract_terminated'].astype(np.uint8)
data_train['distance_to_service_hub'] = data_train['distance_to_service_hub'].astype(np.uint8)
data_train['downtime_tolerance'] = data_train['downtime_tolerance'].astype(np.uint8)
data_train['emergency_service_usage'] = data_train['emergency_service_usage'].astype(bool)
data_train['equipment_complexity_level'] = data_train['equipment_complexity_level'].astype(np.uint8)
data_train['equipment_health_score'] = data_train['equipment_health_score'].astype(np.uint8)
data_train['maintenance_engagement_level'] = data_train['maintenance_engagement_level'].astype(np.uint8)
data_train['monthly_service_fee'] = data_train['monthly_service_fee'].astype(np.int16)
data_train['months_on_current_service_tier'] = data_train['months_on_current_service_tier'].astype(np.uint8)
data_train['months_with_provider'] = data_train['months_with_provider'].astype(np.uint8)
data_train['num_prior_providers'] = data_train['num_prior_providers'].astype(np.uint8)
data_train['onsite_service_satisfaction'] = data_train['onsite_service_satisfaction'].astype(np.uint8)
data_train['overall_contract_satisfaction'] = data_train['overall_contract_satisfaction'].astype(np.uint8)
data_train['preventive_training_sessions_last_year'] = data_train['preventive_training_sessions_last_year'].astype(np.uint8)
data_train['staff_skill_level'] = data_train['staff_skill_level'].astype(np.uint8)
data_train['total_equipment_age'] = data_train['total_equipment_age'].astype(np.uint8)
data_train['years_operating_facility'] = data_train['years_operating_facility'].astype(np.uint8)

In [28]:
data_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1078 entries, 0 to 1086
Data columns (total 20 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   ID                                      1078 non-null   uint16
 1   contract_terminated                     1078 non-null   uint8 
 2   distance_to_service_hub                 1078 non-null   uint8 
 3   downtime_tolerance                      1078 non-null   uint8 
 4   emergency_service_usage                 1078 non-null   bool  
 5   equipment_complexity_level              1078 non-null   uint8 
 6   equipment_health_score                  1078 non-null   uint8 
 7   facility_role_type                      1078 non-null   object
 8   maintenance_engagement_level            1078 non-null   uint8 
 9   monthly_service_fee                     1078 non-null   int16 
 10  months_on_current_service_tier          1078 non-null   uint8 
 11  months_wi

In [29]:
data_train['service_package_tier'].unique()

array(['Silver', 'Bronze', 'Gold', 'Enterprise'], dtype=object)

In [30]:
data_train = pd.concat([data_train, pd.get_dummies(data_train['service_package_tier'], prefix='spt', drop_first=True)], axis=1)

In [31]:
data_train.drop(columns=['service_package_tier'], inplace = True)
data_train

,ID,contract_terminated,distance_to_service_hub,downtime_tolerance,emergency_service_usage,equipment_complexity_level,equipment_health_score,facility_role_type,maintenance_engagement_level,monthly_service_fee,...,num_prior_providers,onsite_service_satisfaction,overall_contract_satisfaction,preventive_training_sessions_last_year,staff_skill_level,total_equipment_age,years_operating_facility,spt_Enterprise,spt_Gold,spt_Silver
0,1,0,3,3,True,1,3,PrecisionManufacturing,2,2177,...,3,3,4,6,3,18,45,False,False,True
1,2,0,3,3,True,2,4,PharmaceuticalProduction,4,6842,...,6,4,1,3,2,24,22,False,False,True
2,3,0,8,3,True,2,3,HeavyIndustrialProcessing,3,5121,...,3,2,1,3,5,18,20,False,False,True
3,4,0,8,3,True,3,3,AdvancedAutomationCenter,2,13757,...,2,4,2,5,1,27,26,False,False,True
4,5,1,9,3,True,1,3,PackagingAndFinishing,2,4400,...,3,1,1,2,2,17,11,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082,1083,0,26,3,True,5,4,LogisticsAndDistribution,2,19586,...,1,3,4,3,5,47,41,False,False,True
1083,1084,0,11,3,True,4,4,LogisticsAndDistribution,3,16032,...,3,2,4,2,4,37,40,False,False,True
1084,1085,0,14,3,True,1,3,PrecisionManufacturing,3,2451,...,4,1,3,2,3,24,27,False,False,True
1085,1086,1,10,4,True,1,4,PrecisionManufacturing,2,1859,...,1,1,2,2,3,12,5,False,False,False


In [32]:
data_train['facility_role_type'].unique()

array(['PrecisionManufacturing', 'PharmaceuticalProduction',
       'HeavyIndustrialProcessing', 'AdvancedAutomationCenter',
       'PackagingAndFinishing', 'LogisticsAndDistribution',
       'TestingAndCalibration', 'AssemblyFacility', 'Other'], dtype=object)

In [33]:
data_train = pd.concat([data_train, pd.get_dummies(data_train['facility_role_type'], prefix='frt', drop_first=True)], axis=1)

In [34]:
data_train.drop(columns=['facility_role_type'], inplace = True)
data_train

,ID,contract_terminated,distance_to_service_hub,downtime_tolerance,emergency_service_usage,equipment_complexity_level,equipment_health_score,maintenance_engagement_level,monthly_service_fee,months_on_current_service_tier,...,spt_Gold,spt_Silver,frt_AssemblyFacility,frt_HeavyIndustrialProcessing,frt_LogisticsAndDistribution,frt_Other,frt_PackagingAndFinishing,frt_PharmaceuticalProduction,frt_PrecisionManufacturing,frt_TestingAndCalibration
0,1,0,3,3,True,1,3,2,2177,0,...,False,True,False,False,False,False,False,False,True,False
1,2,0,3,3,True,2,4,4,6842,4,...,False,True,False,False,False,False,False,True,False,False
2,3,0,8,3,True,2,3,3,5121,0,...,False,True,False,True,False,False,False,False,False,False
3,4,0,8,3,True,3,3,2,13757,8,...,False,True,False,False,False,False,False,False,False,False
4,5,1,9,3,True,1,3,2,4400,2,...,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1082,1083,0,26,3,True,5,4,2,19586,6,...,False,True,False,False,True,False,False,False,False,False
1083,1084,0,11,3,True,4,4,3,16032,9,...,False,True,False,False,True,False,False,False,False,False
1084,1085,0,14,3,True,1,3,3,2451,8,...,False,True,False,False,False,False,False,False,True,False
1085,1086,1,10,4,True,1,4,2,1859,1,...,False,False,False,False,False,False,False,False,True,False


In [35]:
data_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1078 entries, 0 to 1086
Data columns (total 29 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   ID                                      1078 non-null   uint16
 1   contract_terminated                     1078 non-null   uint8 
 2   distance_to_service_hub                 1078 non-null   uint8 
 3   downtime_tolerance                      1078 non-null   uint8 
 4   emergency_service_usage                 1078 non-null   bool  
 5   equipment_complexity_level              1078 non-null   uint8 
 6   equipment_health_score                  1078 non-null   uint8 
 7   maintenance_engagement_level            1078 non-null   uint8 
 8   monthly_service_fee                     1078 non-null   int16 
 9   months_on_current_service_tier          1078 non-null   uint8 
 10  months_with_provider                    1078 non-null   uint8 
 11  num_prior

In [36]:
X = data_train.drop(columns=['contract_terminated'])
y = data_train['contract_terminated']

In [37]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 1/3, random_state=0)

In [38]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

In [43]:
model = Sequential()
model.add(Dense(units=512, activation='relu'))
model.add(Dense(units=10, activation = 'softmax'))
model.compile(loss= 'categorical_crossentropy', metrics= ['accuracy'], optimizer=keras.optimizers.SGD(learning_rate=0.05, momentum=0.4, nesterov=True))

In [44]:
model.fit(X_train_scaled, y_train)

ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 1), output.shape=(None, 10)

In [46]:
history = model.fit(X_train_scaled, y_train, batch_size=128, epochs=30, validation_data=(X_val, y_val))

Epoch 1/30


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.

In [90]:
X_val_scaled = scaler.transform(X_val)
model.score(X_val_scaled, y_val)

0.7222222222222222

In [91]:
y_predicted = model.predict(X_val_scaled)

In [92]:
confusion_matrix(y_predicted, y_val)

array([[221,  18],
       [ 82,  39]])

In [93]:
print(classification_report(y_predicted, y_val))

              precision    recall  f1-score   support

           0       0.73      0.92      0.82       239
           1       0.68      0.32      0.44       121

    accuracy                           0.72       360
   macro avg       0.71      0.62      0.63       360
weighted avg       0.71      0.72      0.69       360



In [94]:
y_predicted

array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1,
       1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0,
       0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0,
       1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0,
       0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1,
       0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0,
       0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1,

In [95]:
data_test = pd.read_csv('IndustrialMaintenanceClientRetention_Testing.csv')
data_test.head()

,ID,distance_to_service_hub,downtime_tolerance,emergency_service_usage,equipment_complexity_level,equipment_health_score,facility_role_type,maintenance_engagement_level,monthly_service_fee,months_on_current_service_tier,months_with_provider,num_prior_providers,onsite_service_satisfaction,overall_contract_satisfaction,preventive_training_sessions_last_year,service_package_tier,staff_skill_level,total_equipment_age,years_operating_facility
0,1,2,3,No,1,3,PrecisionManufacturing,2,4968,7,9,0,4,1,2,Bronze,4,21,16
1,2,29,4,No,1,3,TestingAndCalibration,3,3907,2,6,1,2,1,2,Enterprise,1,17,10
2,3,1,4,No,1,4,Other,3,2064,3,5,0,3,3,3,Silver,3,17,16
3,4,26,3,No,2,3,AssemblyFacility,2,7991,2,2,8,2,2,3,Bronze,2,17,32
4,5,4,3,Yes,2,3,AssemblyFacility,2,5828,7,8,1,4,4,0,Bronze,4,19,12


In [96]:
data_test.dropna(inplace=True)

In [97]:
medians = data_test['months_with_provider'].median()
medians

np.float64(5.0)

In [98]:
data_test['months_with_provider'] = data_test['months_with_provider'].replace(9999999, medians)

In [99]:
data_test['ID'] = data_test['ID'].astype(np.uint16)
data_test['distance_to_service_hub'] = data_test['distance_to_service_hub'].astype(np.uint8)
data_test['downtime_tolerance'] = data_test['downtime_tolerance'].astype(np.uint8)
data_test['emergency_service_usage'] = data_test['emergency_service_usage'].astype(bool)
data_test['equipment_complexity_level'] = data_test['equipment_complexity_level'].astype(np.uint8)
data_test['equipment_health_score'] = data_test['equipment_health_score'].astype(np.uint8)
data_test['maintenance_engagement_level'] = data_test['maintenance_engagement_level'].astype(np.uint8)
data_test['monthly_service_fee'] = data_test['monthly_service_fee'].astype(np.int16)
data_test['months_on_current_service_tier'] = data_test['months_on_current_service_tier'].astype(np.uint8)
data_test['months_with_provider'] = data_test['months_with_provider'].astype(np.uint8)
data_test['num_prior_providers'] = data_test['num_prior_providers'].astype(np.uint8)
data_test['onsite_service_satisfaction'] = data_test['onsite_service_satisfaction'].astype(np.uint8)
data_test['overall_contract_satisfaction'] = data_test['overall_contract_satisfaction'].astype(np.uint8)
data_test['preventive_training_sessions_last_year'] = data_test['preventive_training_sessions_last_year'].astype(np.uint8)
data_test['staff_skill_level'] = data_test['staff_skill_level'].astype(np.uint8)
data_test['total_equipment_age'] = data_test['total_equipment_age'].astype(np.uint8)
data_test['years_operating_facility'] = data_test['years_operating_facility'].astype(np.uint8)

In [100]:
data_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 363 entries, 0 to 362
Data columns (total 19 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   ID                                      363 non-null    uint16
 1   distance_to_service_hub                 363 non-null    uint8 
 2   downtime_tolerance                      363 non-null    uint8 
 3   emergency_service_usage                 363 non-null    bool  
 4   equipment_complexity_level              363 non-null    uint8 
 5   equipment_health_score                  363 non-null    uint8 
 6   facility_role_type                      363 non-null    object
 7   maintenance_engagement_level            363 non-null    uint8 
 8   monthly_service_fee                     363 non-null    int16 
 9   months_on_current_service_tier          363 non-null    uint8 
 10  months_with_provider                    363 non-null    uint8 
 11  num_pr

In [101]:
data_test['service_package_tier'].unique()

array(['Bronze', 'Enterprise', 'Silver', 'Gold'], dtype=object)

In [102]:
data_test = pd.concat([data_test, pd.get_dummies(data_test['service_package_tier'], prefix='spt', drop_first=True)], axis=1)

In [103]:
data_test.drop(columns=['service_package_tier'], inplace = True)
data_test

,ID,distance_to_service_hub,downtime_tolerance,emergency_service_usage,equipment_complexity_level,equipment_health_score,facility_role_type,maintenance_engagement_level,monthly_service_fee,months_on_current_service_tier,...,num_prior_providers,onsite_service_satisfaction,overall_contract_satisfaction,preventive_training_sessions_last_year,staff_skill_level,total_equipment_age,years_operating_facility,spt_Enterprise,spt_Gold,spt_Silver
0,1,2,3,True,1,3,PrecisionManufacturing,2,4968,7,...,0,4,1,2,4,21,16,False,False,False
1,2,29,4,True,1,3,TestingAndCalibration,3,3907,2,...,1,2,1,2,1,17,10,True,False,False
2,3,1,4,True,1,4,Other,3,2064,3,...,0,3,3,3,3,17,16,False,False,True
3,4,26,3,True,2,3,AssemblyFacility,2,7991,2,...,8,2,2,3,2,17,32,False,False,False
4,5,4,3,True,2,3,AssemblyFacility,2,5828,7,...,1,4,4,0,4,19,12,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
358,359,2,3,True,2,3,AssemblyFacility,3,5677,8,...,3,2,2,4,2,26,26,False,False,True
359,360,17,2,True,1,3,Other,3,2741,7,...,0,2,3,2,4,19,12,False,False,True
360,361,1,2,True,1,3,TestingAndCalibration,3,3229,2,...,4,1,4,2,3,18,11,False,False,True
361,362,9,2,True,2,3,TestingAndCalibration,3,5346,1,...,4,4,1,3,4,22,20,False,False,True


In [104]:
data_test['facility_role_type'].unique()

array(['PrecisionManufacturing', 'TestingAndCalibration', 'Other',
       'AssemblyFacility', 'LogisticsAndDistribution',
       'AdvancedAutomationCenter', 'HeavyIndustrialProcessing',
       'PharmaceuticalProduction', 'PackagingAndFinishing'], dtype=object)

In [105]:
data_test = pd.concat([data_test, pd.get_dummies(data_test['facility_role_type'], prefix='frt', drop_first=True)], axis=1)

In [106]:
data_test.drop(columns=['facility_role_type'], inplace = True)
data_test

,ID,distance_to_service_hub,downtime_tolerance,emergency_service_usage,equipment_complexity_level,equipment_health_score,maintenance_engagement_level,monthly_service_fee,months_on_current_service_tier,months_with_provider,...,spt_Gold,spt_Silver,frt_AssemblyFacility,frt_HeavyIndustrialProcessing,frt_LogisticsAndDistribution,frt_Other,frt_PackagingAndFinishing,frt_PharmaceuticalProduction,frt_PrecisionManufacturing,frt_TestingAndCalibration
0,1,2,3,True,1,3,2,4968,7,9,...,False,False,False,False,False,False,False,False,True,False
1,2,29,4,True,1,3,3,3907,2,6,...,False,False,False,False,False,False,False,False,False,True
2,3,1,4,True,1,4,3,2064,3,5,...,False,True,False,False,False,True,False,False,False,False
3,4,26,3,True,2,3,2,7991,2,2,...,False,False,True,False,False,False,False,False,False,False
4,5,4,3,True,2,3,2,5828,7,8,...,False,False,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
358,359,2,3,True,2,3,3,5677,8,11,...,False,True,True,False,False,False,False,False,False,False
359,360,17,2,True,1,3,3,2741,7,7,...,False,True,False,False,False,True,False,False,False,False
360,361,1,2,True,1,3,3,3229,2,3,...,False,True,False,False,False,False,False,False,False,True
361,362,9,2,True,2,3,3,5346,1,7,...,False,True,False,False,False,False,False,False,False,True


In [107]:
data_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 363 entries, 0 to 362
Data columns (total 28 columns):
 #   Column                                  Non-Null Count  Dtype 
---  ------                                  --------------  ----- 
 0   ID                                      363 non-null    uint16
 1   distance_to_service_hub                 363 non-null    uint8 
 2   downtime_tolerance                      363 non-null    uint8 
 3   emergency_service_usage                 363 non-null    bool  
 4   equipment_complexity_level              363 non-null    uint8 
 5   equipment_health_score                  363 non-null    uint8 
 6   maintenance_engagement_level            363 non-null    uint8 
 7   monthly_service_fee                     363 non-null    int16 
 8   months_on_current_service_tier          363 non-null    uint8 
 9   months_with_provider                    363 non-null    uint8 
 10  num_prior_providers                     363 non-null    uint8 
 11  onsite

In [108]:
data_test.describe()

,ID,distance_to_service_hub,downtime_tolerance,equipment_complexity_level,equipment_health_score,maintenance_engagement_level,monthly_service_fee,months_on_current_service_tier,months_with_provider,num_prior_providers,onsite_service_satisfaction,overall_contract_satisfaction,preventive_training_sessions_last_year,staff_skill_level,total_equipment_age,years_operating_facility
count,363.000000,363.000000,363.000000,363.000000,363.000000,363.000000,363.000000,363.000000,363.000000,363.000000,363.000000,363.000000,363.000000,363.000000,363.000000,363.000000
mean,182.000000,9.082645,2.768595,2.022039,3.134986,2.741047,6214.371901,4.347107,6.917355,2.630854,2.765840,2.710744,2.807163,2.892562,21.994490,22.435262
std,104.933312,8.092613,0.729342,1.066600,0.342181,0.727628,4481.525948,3.623993,5.864850,2.514343,1.088737,1.075375,1.354858,0.992805,7.897127,9.299086
min,1.000000,1.000000,1.000000,1.000000,3.000000,1.000000,1052.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,11.000000,4.000000
25%,91.500000,2.000000,2.000000,1.000000,3.000000,2.000000,2922.500000,2.000000,3.000000,1.000000,2.000000,2.000000,2.000000,2.000000,17.000000,15.000000
50%,182.000000,7.000000,3.000000,2.000000,3.000000,3.000000,4663.000000,3.000000,5.000000,1.000000,3.000000,3.000000,3.000000,3.000000,20.000000,21.000000
75%,272.500000,13.000000,3.000000,3.000000,3.000000,3.000000,7497.000000,7.000000,9.000000,4.000000,4.000000,4.000000,3.000000,4.000000,25.000000,29.000000
max,363.000000,29.000000,4.000000,5.000000,4.000000,4.000000,19999.000000,18.000000,33.000000,9.000000,4.000000,4.000000,6.000000,5.000000,48.000000,45.000000


In [109]:
X_test_scaled = scaler.transform(data_test)

In [110]:
y_predicted_test = model.predict(X_test_scaled)

In [111]:
pd.DataFrame(y_predicted_test, columns=["contract_terminated"]).to_csv("predictions2.csv", index=False)

In [112]:
df = pd.read_csv('predictions2.csv')
df.insert(0, "ID", range(1, len(df)+1))
pd.DataFrame(df, columns=["ID","contract_terminated"]).to_csv("predictions_2.csv", index=False)